In [1]:
import os
import pandas as pd
from pathlib import Path
from bs4 import BeautifulSoup
import re
import dateparser

In [ ]:
def carregar_historico(base_path):
    base_dir = Path(base_path) / "histórico"
    arquivo = base_dir / "histórico-de-visualização.html"

    with open(arquivo, "r", encoding="utf-8") as f:
        soup = BeautifulSoup(f, "html.parser")

    dados = []

    blocos = soup.find_all("div", class_="content-cell")

    print("Blocos encontrados:", len(blocos))

    for bloco in blocos:
        link = bloco.find("a")
        if not link:
            continue

        titulo = link.text.strip()
        url = link["href"]

        texto = bloco.get_text(separator=" ", strip=True)

        partes = texto.split(" em ")
        data = partes[-1] if len(partes) > 1 else None

        dados.append([titulo, url, data])

    df = pd.DataFrame(dados, columns=["titulo", "url", "data"])
    return df

In [3]:
BASE_PATH = "../data/Nacarelli/YouTube e YouTube Music"

In [4]:
df = carregar_historico(BASE_PATH)

print("Shape antes da conversão:", df.shape)

# Converter data corretamente
df["data"] = df["data"].apply(
    lambda x: dateparser.parse(str(x), languages=["pt"])
)

print("Datas válidas:", df["data"].notna().sum())

# Agora sim remover inválidas
df = df.dropna(subset=["data"])

# Ordenar
df = df.sort_values("data").reset_index(drop=True)

print("Shape depois da limpeza:", df.shape)

df.head()

Shape antes da conversão: (17669, 4)
Datas válidas: 5130
Shape depois da limpeza: (5130, 4)


,tipo,titulo,url,data
0,Searched for,corinthians x ponte preta,https://www.youtube.com/results?search_query=c...,2022-03-12 20:59:54-03:00
1,Searched for,corinthians x ponte preta gol,https://www.youtube.com/results?search_query=c...,2022-03-12 21:03:42-03:00
2,Searched for,reu acusado,https://www.youtube.com/results?search_query=r...,2022-03-13 07:50:44-03:00
3,Searched for,corinthians x ponte preta,https://www.youtube.com/results?search_query=c...,2022-03-13 08:00:59-03:00
4,Searched for,corinthians x ponte preta,https://www.youtube.com/results?search_query=c...,2022-03-13 15:18:25-03:00


In [5]:
df["proximo_video"] = df["titulo"].shift(-1)

df_modelo = df[["titulo", "proximo_video"]].dropna()

df_modelo.head()

,titulo,proximo_video
0,corinthians x ponte preta,corinthians x ponte preta gol
1,corinthians x ponte preta gol,reu acusado
2,reu acusado,corinthians x ponte preta
3,corinthians x ponte preta,corinthians x ponte preta
4,corinthians x ponte preta,skins de youtubers


In [6]:
df = df[df["tipo"].str.contains("assistiu", case=False, na=False)]

In [8]:
# df = carregar_historico(BASE_PATH)
# print(df.shape)
# df.head()

In [9]:
print(df.shape)
print(df["data"].head())
print(df["data"].dtype)

(0, 5)
Series([], Name: data, dtype: object)
object


In [10]:
df["data"] = df["data"].astype(str)

# Converter usando parser inteligente
df["data"] = df["data"].apply(
    lambda x: dateparser.parse(x, languages=["pt"])
)

print("Datas válidas:", df["data"].notna().sum())
print("Shape:", df.shape)

Datas válidas: 0
Shape: (0, 5)


In [11]:
df["proximo_video"] = df["titulo"].shift(-1)
df_modelo = df[["titulo", "proximo_video"]].dropna()

In [12]:
df_modelo.to_csv("df_modelo.csv", index=False)